In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities_copy import extractor
import uproot
import awkward as ak    

x_MH175=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH175_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH175_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]
#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH175[mask]

from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 125 < x < 225]

def voigt2(x, norm, mu, sigma, gamma, norm2, mu2, sigma2, gamma2):
    return voigt_profile(x-mu, sigma, gamma) * norm + norm2*voigt_profile(x-mu2, sigma2, gamma2)

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt2)

m_voigt=Minuit(ls_voigt,  norm=1, mu=175, sigma=5, gamma=1, norm2=1, mu2=175, sigma2=5, gamma2=0.001)
m_voigt.limits["mu"]= (125, 225)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.fixed["gamma2"]= True

m_voigt.migrad()

/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 56.99 (χ²/ndof = 1.3)      │              Nfcn = 899              │
│ EDM = 4.88e-07 (Goal: 0.0002)    │            time = 0.2 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name   │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm   │   0.59    │   0.08    │            │            │         │         │       │
│ 1 │ mu     │  180.80   │   0.18    │            │            │   125   │   225   │       │
│ 2 │ sigma  │   10.4    │    0.5    │            │            │   0.1   │   20    │       │
│ 3 │ gamma  │    1.8    │    1.5    │            │            │  0.01   │   10    │       │
│ 4 │ norm2  │   0.44    │   0.07    │            │            │         │         │       │
│ 5 │ mu2    │   168.9   │    1.9    │            │            │         │         │       │
│ 6 │ sigma2 │   22.2    │    0.8    │            │            │         │         │       │
│ 7 │ gamma2 │  1.00e-3  │  0.01e-3  │            │            │         │         │  yes  │
└───┴────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌────────┬─────────────────────────────────────────────────────────────────┐
│        │    norm      mu   sigma   gamma   norm2     mu2  sigma2  gamma2 │
├────────┼─────────────────────────────────────────────────────────────────┤
│   norm │ 0.00721   0.003  -0.029   0.127  -0.006  -0.158  -0.053   0.000 │
│     mu │   0.003  0.0341  -0.044   0.096  -0.002  -0.077  -0.085   0.000 │
│  sigma │  -0.029  -0.044   0.212   -0.62   0.024    0.61    0.33    0.00 │
│  gamma │   0.127   0.096   -0.62    2.41  -0.109    -2.8    -1.1     0.0 │
│  norm2 │  -0.006  -0.002   0.024  -0.109 0.00539   0.137   0.045   0.000 │
│    mu2 │  -0.158  -0.077    0.61    -2.8   0.137    3.55     1.2       0 │
│ sigma2 │  -0.053  -0.085    0.33    -1.1   0.045     1.2   0.625     0.0 │
│ gamma2 │   0.000   0.000    0.00     0.0   0.000       0     0.0       0 │
└────────┴─────────────────────────────────────────────────────────────────┘

In [2]:
fit_MH175_values={}
fit_MH175_errors={}

fit_values={'MH175': fit_MH175_values,}
fit_errors={'MH175_errors': fit_MH175_errors}



for param in m_voigt.parameters:
    fit_MH175_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH175_errors[error] = m_voigt.errors[error]

print(fit_MH175_values)
print(fit_MH175_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH175"]=fit_MH175_values
results["MH175_errors"]=fit_MH175_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH175"]=fit_MH175_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH175_errors"]=fit_MH175_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  

{'norm': 0.5906981106772375, 'mu': 180.79793247155462, 'sigma': 10.447570853158268, 'gamma': 1.7847597884132067, 'norm2': 0.43520668578981375, 'mu2': 168.9379447383002, 'sigma2': 22.155413761653445, 'gamma2': 0.001}
{'norm': 0.08491919482396466, 'mu': 0.18468302201662823, 'sigma': 0.4604526001389706, 'gamma': 1.508732875922611, 'norm2': 0.0734076416742038, 'mu2': 1.8835641458979233, 'sigma2': 0.7904267064449214, 'gamma2': 1e-05}
